# ReFactX: New Features Interactive Test

This notebook tests the three new features with custom prompts that force the model to use them:
1. **Count Branches Tool** - `count_branches: <Subject> <relation>` inline tool for counting
2. **Sentinel for Exhausted Relations** - `no further records>` signal when all objects are generated
3. **Thinking Cache Reset** - `</think>` reset allowing triple reuse after thinking

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import time
from transformers.generation.logits_process import LogitsProcessorList
from transformers import AutoProcessor, AutoModelForImageTextToText, TextStreamer
from transformers import ProcessorMixin

import refactx
from refactx.generate import (
    patch_model, CONSTRAINED_STATES,
    FactGeneration, CountBranchesGeneration,
    ConstrainedLogitsProcessor,
)

## Configuration

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
MODEL = 'Qwen/Qwen3.5-4B'
INDEX = os.environ.get('POSTGRES_URL', '../indexes/simple_index.txt.gz')

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

## Load Model and Index

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL)
model = AutoModelForImageTextToText.from_pretrained(MODEL, device_map='auto')
tokenizer = processor
streamer = TextStreamer(tokenizer.tokenizer)

In [ ]:
index = refactx.load_index(INDEX, tokenizer=tokenizer)

In [ ]:
refactx.patch_model(model)

## Custom Prompt Builder

These prompts include few-shot examples that teach the model to:
- Use `count_branches: <Subject> <relation>` for counting (Feat 1)
- Recognize `no further records>` when a relation is exhausted (Feat 2)
- Use `<think>` blocks in reasoning (Feat 3)

In [ ]:
# Feature-forcing system prompt
SYSTEM_PROMPT_FEATURES = (
    'You are a helpful question-answering assistant that bases its answers on facts from a knowledge base.\n'
    '\n'
    '## Fact retrieval\n'
    'To obtain facts, use the Fact: command (e.g. Fact: <Danny Boyle> <date of birth> <1956-10-20T00:00:00Z> .).\n'
    'Facts are always reliable. You must support all your claims with facts.\n'
    '\n'
    '## Counting tool\n'
    'When you need to count how many objects exist for a subject-relation prefix, use the count_branches tool.\n'
    'Write exactly: count_branches: <Subject> <relation>\n'
    'The system will reply with = <number>. Use that number in your answer.\n'
    'Example: count_branches: <Spain> <shares border with>  -> the system replies = 5\n'
    '\n'
    '## Exhausted records\n'
    'When a Fact: returns no further records>, it means all objects for that subject-relation have been generated.\n'
    'Stop requesting facts with that prefix and use what you have.\n'
    '\n'
    '## Reasoning\n'
    'You may think step by step before answering. Keep your reasoning concise.\n'
    'After reasoning, give a concise answer prefixed with Answer:.'
)

# Few-shot: count_branches tool (Feat 1)
FS_COUNT_SPAIN = [
    {'role': 'user', 'content': 'How many countries share a border with Spain?'},
    {'role': 'assistant', 'content': (
        '<think>\n'
        'I need to count the countries that share a land border with Spain. '
        'I can use the count_branches tool to count them efficiently.\n'
        '</think>\n'
        '\n'
        'count_branches: <Spain> <shares border with>\n'
        '= 5\n'
        'The tool tells me there are 5 countries that share a border with Spain.\n'
        '\n'
        'Answer: 5.'
    )}
]

# Few-shot: exhausted relations (Feat 2)
FS_SENTINEL = [
    {'role': 'user', 'content': 'Which countries share a border with France?'},
    {'role': 'assistant', 'content': (
        '<think>\n'
        'I need to find all countries that share a border with France.\n'
        '</think>\n'
        '\n'
        'Fact: <France> <shares border with> <Belgium> .\n'
        'Fact: <France> <shares border with> <Germany> .\n'
        'Fact: <France> <shares border with> <Spain> .\n'
        'Fact: <France> <shares border with> <Italy> .\n'
        'Fact: <France> <shares border with> <Switzerland> .\n'
        'Fact: <France> <shares border with> <Luxembourg> .\n'
        'Fact: <France> <shares border with> <no further records> .\n'
        'All objects for France shares border with have been exhausted. '
        'I found 6 countries.\n'
        '\n'
        'Answer: Belgium, Germany, Spain, Italy, Switzerland, Luxembourg.'
    )}
]

# Few-shot: thinking + facts (Feat 3)
FS_THINKING = [
    {'role': 'user', 'content': 'Was Danny Boyle born before Napoleon Bonaparte?'},
    {'role': 'assistant', 'content': (
        '<think>\n'
        'I need to compare the birth dates of Danny Boyle and Napoleon Bonaparte.\n'
        'Let me get their birth dates.\n'
        '</think>\n'
        '\n'
        'Fact: <Danny Boyle> <date of birth> <1956-10-20T00:00:00Z> .\n'
        'Fact: <Napoleon Bonaparte> <date of birth> <1769-08-15T00:00:00Z> .\n'
        'Danny Boyle was born in 1956, while Napoleon was born in 1769. So Danny Boyle was born after Napoleon.\n'
        '\n'
        'Answer: No.'
    )}
]

def build_prompt(question, few_shot):
    prompt = [{'role': 'system', 'content': SYSTEM_PROMPT_FEATURES}]
    prompt.extend(few_shot)
    prompt.append({'role': 'user', 'content': question})
    return prompt

print('Custom prompts defined.')

## Helper Function

In [ ]:
def _tokenize(tok, text):
    if isinstance(tok, ProcessorMixin):
        return tok.tokenizer(text, return_tensors='pt')
    return tok(text, return_tensors='pt')

def _decode(tok, ids):
    if isinstance(tok, ProcessorMixin):
        return tok.tokenizer.decode(ids, skip_special_tokens=True)
    return tok.decode(ids, skip_special_tokens=True)

def ask(question, max_new_tokens=800, sentinel=True, few_shot=None):
    """Ask a question with the new PatternConstrainedGeneration architecture.

    Creates a ConstrainedLogitsProcessor, registers both Fact: and count_branches: patterns,
    then runs generation.
    """
    if few_shot is None:
        few_shot = []
    prompted = build_prompt(question, few_shot)
    full_prompt = tokenizer.apply_chat_template(prompted, tokenize=False, add_generation_prompt=True)
    inputs = _tokenize(tokenizer, full_prompt).to(model.device)

    # Fresh ConstrainedLogitsProcessor each call
    CONSTRAINED_STATES.__init__(
        'auto', num_beams=1, num_batches=1, debug_tokenizer=tokenizer)
    proc = ConstrainedLogitsProcessor(
        states=CONSTRAINED_STATES, tokenizer=tokenizer)
    proc.add_pattern('Fact:', FactGeneration,
                     index=index, sentinel=sentinel, eot=None)
    proc.add_pattern('count_branches:', CountBranchesGeneration,
                     kb_index=index)
    logits_processor = LogitsProcessorList([proc])

    model.eval()
    start = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            logits_processor=logits_processor,
            max_new_tokens=max_new_tokens,
            streamer=streamer,
            do_sample=False,
            num_beams=1,
            num_return_sequences=1,
            use_cache=True,
        )
    elapsed = time.time() - start

    state = CONSTRAINED_STATES.states[0][0]
    text = _decode(tokenizer, out[0][inputs.input_ids.shape[1]:])
    facts = state.generated_triples
    facts_str = [_decode(tokenizer, t) for t in facts]

    for i, f in enumerate(facts_str):
        print(f'  {i}: {f}')

    # Show generation history
    for i, g in enumerate(state.generation_history):
        cls_name = type(g).__name__
        if hasattr(g, 'completed_with_sentinel'):
            print(f'  history[{i}]: {cls_name} sentinel={g.completed_with_sentinel}')
        elif hasattr(g, 'called'):
            print(f'  history[{i}]: {cls_name} called={g.called}')
        else:
            print(f'  history[{i}]: {cls_name}')

    print(f'Elapsed: {elapsed:.2f}s')
    return text, facts_str

## Feat 1: Count Branches Tool

The prompt teaches the model to use `count_branches: <Subject> <relation>`.
The `CountBranchesGeneration` pattern intercepts this and counts KB entries.

**Test**: Ask a "how many" question with the count_branches few-shot example.

In [ ]:
type(index)

In [ ]:
#import pdb
#pdb.set_trace()
text, facts = ask(
    'How many and which countries share a border with Brazil?',
    few_shot=FS_COUNT_SPAIN)

## Feat 2: Sentinel for Exhausted Relations

The prompt teaches the model that `no further records>` means a relation is exhausted.
When `sentinel=True`, the constrained processor emits this message automatically.

**Test**: Ask a question that requires exhausting a subject-relation's objects.

In [ ]:
import pdb
pdb.set_trace()
text, facts = ask(
    'Which countries share a border with France?',
    sentinel=True,
    max_new_tokens=1200,
    few_shot=FS_SENTINEL)

## Feat 3: Thinking Cache Reset

The prompt teaches the model to use `<think>` blocks for reasoning.
After `</think>`, the duplicate cache resets, allowing the model to generate facts
that may overlap with facts from the thinking block.

**Test**: Ask a comparison question that triggers thinking + facts.

In [ ]:
text, facts = ask(
    'Was Danny Boyle born before Napoleon Bonaparte?',
    sentinel=True,
    max_new_tokens=1200,
    few_shot=FS_THINKING)

## Combined Test: All Features Together

This test combines all three features in a single prompt:
- The model uses `<think>` to reason
- It calls `count_branches:` to count entities
- It handles `no further records>` when relations are exhausted

In [ ]:
# Combined few-shot with all features
FS_COMBINED = FS_COUNT_SPAIN + FS_SENTINEL + FS_THINKING

text, facts = ask(
    'How many countries border Italy?',
    max_new_tokens=1200,
    few_shot=FS_COMBINED)

## Interactive Testing

Edit the question and few-shot examples below to test interactively.
You can combine any of the three feature prompts.

In [ ]:
# Pick which few-shot examples to include
USE_COUNT_SHOT = True    # Feat 1: count_branches
USE_SENTINEL_SHOT = True # Feat 2: no further records
USE_THINKING_SHOT = True # Feat 3: think blocks

active_few_shot = []
if USE_COUNT_SHOT:
    active_few_shot.extend(FS_COUNT_SPAIN)
if USE_SENTINEL_SHOT:
    active_few_shot.extend(FS_SENTINEL)
if USE_THINKING_SHOT:
    active_few_shot.extend(FS_THINKING)

# Change this question to test interactively
MY_QUESTION = 'How many countries share a border with Germany?'

text, facts = ask(
    MY_QUESTION,
    max_new_tokens=1200,
    few_shot=active_few_shot)